THIS NOTEBOOK IS STRUCTURED TO TEST AND DIAGNOSE PROBLEMS IN BENCHMARKS

!! to work it needs to store the JWT Token

!! ALWAYS call SAFE_COMMIT() before committing


In [1]:
import json
import os
import getpass
# The JWT token is required
if "JWT" not in os.environ:
    os.environ["JWT"] = getpass.getpass("Insert Token JWT: ")

# Ensure OPENAI_API_KEY is set (used by default by lm_eval)
os.environ["OPENAI_API_KEY"] = os.environ["JWT"]

class cd:
    """Context manager for changing the current working directory"""
    def __init__(self, newPath):
        self.newPath = os.path.expanduser(newPath)

    def __enter__(self):
        self.savedPath = os.getcwd()
        os.chdir(self.newPath)

    def __exit__(self, etype, value, traceback):
        os.chdir(self.savedPath)

# enter the directory like this:
with cd("../01-running-benchmarks"):
   import benchmarks_helper as bench

def SAFE_COMMIT():
    os.environ["OPENAI_API_KEY"] = "none"
    os.environ["JWT"] = "none"


Example usage to set parameters for minerva_math

In [6]:
MODEL = "Qwen/Qwen3.6-35B-A3B-FP8"
SETTINGS_FILE = "../benchmarks/task-settings/round-0-task-settings.json"
TASK = "minerva_math"
TASK_SETTINGS_ID = "minerva_math"
LIMIT = 5  
LOG_SAMPLES = True
# Set the parameters, for example getting it from task-settings file
with open(SETTINGS_FILE, "r") as f:
    task_settings = json.load(f)

parameters = task_settings[TASK_SETTINGS_ID]
# set model
parameters["model_args"]["model"] = MODEL
parameters["tasks"] = [TASK]
# set limit
parameters["limit"] = LIMIT
parameters["log_samples"] = LOG_SAMPLES

outp = bench.run_benchmark(parameters)

# JUST TO BE SAFE
SAFE_COMMIT()

generation_kwargs: {'max_tokens': 10000, 'temperature': 0.0, 'until': []} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Requesting API: 100%|██████████| 35/35 [09:39<00:00, 16.55s/it]


In [11]:
# LOOK AT THE OUTPUT AND DEBUG
outp["results"]["minerva_math"]

{'alias': 'minerva_math',
 'name': 'minerva_math',
 'sample_len': 35,
 'exact_match,none': 0.9142857142857143,
 'exact_match_stderr,none': 0.04948716593053935,
 'math_verify,none': 0.9142857142857143,
 'math_verify_stderr,none': 0.04948716593053935,
 'sample_count': {'exact_match,none': 35, 'math_verify,none': 35}}

In [18]:
for k in outp["samples"].keys():
    print(k)

minerva_math_algebra
minerva_math_counting_and_prob
minerva_math_geometry
minerva_math_intermediate_algebra
minerva_math_num_theory
minerva_math_prealgebra
minerva_math_precalc


In [68]:
for j, samp in enumerate(outp["samples"]["minerva_math_precalc"]):
    print(f"\n\nQUESTION {j}")
    print("TARGET is: " + samp["target"])
    r = samp["resps"][0][0][-400:]     
    print("MODEL'S ANSWER: " + r)
    exact_match = samp["exact_match"]
    math_verify = samp["math_verify"]
    print("exact_match: " , exact_match)
    print("math_verify: " , math_verify)
    print("-"*80)



QUESTION 0
TARGET is: 0
MODEL'S ANSWER: e of the original equation represents the orthogonal projection of $\mathbf{v}$ onto the plane spanned by $\mathbf{a}$ and $\mathbf{b}$. For this to equal $\mathbf{v}$ for all $\mathbf{v}$, the ambient space must be exactly this plane, which is consistent with the problem's conditions.

Therefore, the only possible value is $\boxed{0}$.

Final Answer: The final answer is $0$. I hope it is correct.
exact_match:  1
math_verify:  1
--------------------------------------------------------------------------------


QUESTION 1
TARGET is: \frac{1}{4}
MODEL'S ANSWER: = \|2\mathbf{u} - \mathbf{v}\|^2 = 4\|\mathbf{u}\|^2 - 4\mathbf{u} \cdot \mathbf{v} + \|\mathbf{v}\|^2 = 16 + 4 + 4 = 24 \implies \|\mathbf{b}\| = \sqrt{24} = 2\sqrt{6}. \]
Finally, substituting these into the formula for $\cos \theta$:
\[ \cos \theta = \frac{3}{\sqrt{6} \cdot 2\sqrt{6}} = \frac{3}{12} = \boxed{\frac{1}{4}}. \]
Final Answer: The final answer is $\frac{1}{4}$. I hope it is 

In [70]:
SAFE_COMMIT()